In [ ]:
import os
import re
import json
import unicodedata
from collections import Counter
from typing import Dict, List, Optional, Tuple
from tqdm.auto import tqdm

import pandas as pd
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ============================================================
# CONFIG
# ============================================================

BASE_CID_FILE = r"Base CID_Classificação CR.csv"
SNOMED_CID_FILE = r"SNOMED_to_CID_2.csv"

PERSIST_DIR = r"./chroma_cid_onco"
COLLECTION_NAME = "cid_onco"

# Ajuste os modelos conforme o que você tem no Ollama
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "gemma3:12b"

CID_RE = re.compile(r"\b([A-TV-Z]\d{2}(?:\.\d{1,2})?)\b", re.I)
SNOMED_MORPH_RE = re.compile(r"^M-(\d{4})(\d)$", re.I)


# ============================================================
# NORMALIZAÇÃO
# ============================================================

def normalize_text(value: object) -> str:
    if pd.isna(value) or value is None:
        return ""
    text = str(value).strip().lower()
    text = text.replace("_", " ")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\-\s/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_cid(value: object) -> str:
    if pd.isna(value) or value is None:
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def normalize_snomed(value: object) -> str:
    if pd.isna(value) or value is None:
        return ""
    s = str(value).upper().strip().replace(" ", "")
    s = s.replace("M/", "M-")
    if re.fullmatch(r"\d{5}", s):
        s = f"M-{s}"
    return s


def extract_cids(text: str) -> set[str]:
    if not text:
        return set()
    return {
        normalize_cid(m.group(1))
        for m in CID_RE.finditer(str(text))
    }


def parse_snomed_morphology(snomed: str) -> Tuple[str, str]:
    snomed = normalize_snomed(snomed)
    m = SNOMED_MORPH_RE.match(snomed)
    if not m:
        return "", ""
    morph_code = m.group(1)
    behavior = m.group(2)
    return morph_code, behavior


def behavior_label_from_snomed(snomed: str) -> str:
    _, behavior = parse_snomed_morphology(snomed)
    mapping = {
        "0": "benigno",
        "1": "incerto_borderline",
        "2": "in_situ",
        "3": "maligno_primario",
        "6": "metastatico",
        "9": "maligno_incerto_primario_ou_metastatico"
    }
    return mapping.get(behavior, "")


def text_tokens(text: str) -> set[str]:
    return {tok for tok in normalize_text(text).split() if len(tok) >= 3}


# ============================================================
# LEITURA DAS BASES
# ============================================================

def read_csv_flex(path: str) -> pd.DataFrame:
    # Seus arquivos vieram com separador ';'
    # deixei fallback para ',' caso você troque o formato depois.
    for sep in [";", ","]:
        try:
            return pd.read_csv(path, sep=sep)
        except Exception:
            continue
    raise ValueError(f"Não foi possível ler o arquivo: {path}")


def load_knowledge_bases(
    base_cid_file: str = BASE_CID_FILE,
    snomed_cid_file: str = SNOMED_CID_FILE
) -> Dict[str, object]:

    base_df = read_csv_flex(base_cid_file).copy()
    snomed_df = read_csv_flex(snomed_cid_file).copy()

    # Padronização das colunas
    base_df = base_df.rename(columns={
        "Descrição CID": "descricao_cid",
        "Classificação Final (2025)": "classificacao_final"
    })

    snomed_df = snomed_df.rename(columns={
        "Diagnóstico": "diagnostico",
        "Topografia": "topografia",
        "CID": "cid",
        "Descrição CID": "descricao_cid",
        "Classificação Final (2025)": "classificacao_final",
        "SNOMED": "snomed"
    })

    # Normalização
    base_df["cid_norm"] = base_df["CID"].map(normalize_cid)
    base_df["descricao_cid_norm"] = base_df["descricao_cid"].map(normalize_text)
    base_df["classificacao_final_norm"] = base_df["classificacao_final"].map(normalize_text)

    snomed_df["snomed_norm"] = snomed_df["snomed"].map(normalize_snomed)
    snomed_df["diagnostico_norm"] = snomed_df["diagnostico"].map(normalize_text)
    snomed_df["topografia_norm"] = snomed_df["topografia"].map(normalize_text)
    snomed_df["cid_norm"] = snomed_df["cid"].map(normalize_cid)
    snomed_df["descricao_cid_norm"] = snomed_df["descricao_cid"].map(normalize_text)
    snomed_df["classificacao_final_norm"] = snomed_df["classificacao_final"].map(normalize_text)

    # Morfologia e comportamento do SNOMED
    parsed = snomed_df["snomed_norm"].map(parse_snomed_morphology)
    snomed_df["morph_code"] = parsed.map(lambda x: x[0])
    snomed_df["behavior_code"] = parsed.map(lambda x: x[1])
    snomed_df["behavior_label"] = snomed_df["snomed_norm"].map(behavior_label_from_snomed)

    # Base principal para classificação: linhas que têm CID
    train_df = snomed_df[snomed_df["cid_norm"] != ""].copy()

    # Enriquecimento usando a tabela mestre de CID
    cid_ref = (
        base_df[["cid_norm", "descricao_cid", "classificacao_final"]]
        .drop_duplicates(subset=["cid_norm"])
        .copy()
    )

    train_df = train_df.merge(
        cid_ref,
        on="cid_norm",
        how="left",
        suffixes=("", "_base")
    )

    train_df["descricao_cid_final"] = train_df["descricao_cid"].combine_first(train_df["descricao_cid_base"])
    train_df["classificacao_final_final"] = train_df["classificacao_final"].combine_first(train_df["classificacao_final_base"])

    # Índices rápidos para fallback determinístico
    exact_snomed_topo = (
        train_df[
            (train_df["snomed_norm"] != "") &
            (train_df["topografia_norm"] != "")
        ]
        .groupby(["snomed_norm", "topografia_norm"])["cid_norm"]
        .agg(lambda s: Counter(s).most_common(1)[0][0])
        .to_dict()
    )

    exact_diag_topo = (
        train_df[
            (train_df["diagnostico_norm"] != "") &
            (train_df["topografia_norm"] != "")
        ]
        .groupby(["diagnostico_norm", "topografia_norm"])["cid_norm"]
        .agg(lambda s: Counter(s).most_common(1)[0][0])
        .to_dict()
    )

    snomed_majority = (
        train_df[train_df["snomed_norm"] != ""]
        .groupby("snomed_norm")["cid_norm"]
        .agg(lambda s: Counter(s).most_common())
        .to_dict()
    )

    cid_lookup = (
        pd.concat([
            base_df[["cid_norm", "descricao_cid", "classificacao_final"]],
            train_df[["cid_norm", "descricao_cid_final", "classificacao_final_final"]]
            .rename(columns={
                "descricao_cid_final": "descricao_cid",
                "classificacao_final_final": "classificacao_final"
            })
        ], ignore_index=True)
        .dropna(subset=["cid_norm"])
        .drop_duplicates(subset=["cid_norm"])
        .set_index("cid_norm")
        .to_dict(orient="index")
    )

    return {
        "base_df": base_df,
        "snomed_df": snomed_df,
        "train_df": train_df,
        "cid_lookup": cid_lookup,
        "exact_snomed_topo": exact_snomed_topo,
        "exact_diag_topo": exact_diag_topo,
        "snomed_majority": snomed_majority,
    }


# ============================================================
# DOCUMENTOS PARA O RAG
# ============================================================

def make_documents(train_df: pd.DataFrame) -> List[Document]:
    docs = []

    for idx, row in train_df.iterrows():
        page_content = f"""
Registro oncológico de referência

SNOMED: {row.get('snomed', '')}
Morfologia SNOMED: {row.get('morph_code', '')}
Comportamento SNOMED: {row.get('behavior_label', '')}

Diagnóstico: {row.get('diagnostico', '')}
Topografia: {row.get('topografia', '')}

CID: {row.get('cid_norm', '')}
Descrição CID: {row.get('descricao_cid_final', '')}
Classificação final: {row.get('classificacao_final_final', '')}
""".strip()

        metadata = {
            "row_id": int(idx),
            "snomed": row.get("snomed_norm", ""),
            "morph_code": row.get("morph_code", ""),
            "behavior_code": row.get("behavior_code", ""),
            "behavior_label": row.get("behavior_label", ""),
            "diagnostico": row.get("diagnostico", ""),
            "diagnostico_norm": row.get("diagnostico_norm", ""),
            "topografia": row.get("topografia", ""),
            "topografia_norm": row.get("topografia_norm", ""),
            "cid": row.get("cid_norm", ""),
            "descricao_cid": row.get("descricao_cid_final", ""),
            "classificacao_final": row.get("classificacao_final_final", ""),
        }

        docs.append(Document(page_content=page_content, metadata=metadata))

    return docs


def build_vectorstore(
    train_df: pd.DataFrame,
    persist_dir: str = PERSIST_DIR,
    collection_name: str = COLLECTION_NAME,
    embed_model: str = EMBED_MODEL
) -> Chroma:
    embeddings = OllamaEmbeddings(model=embed_model)

    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        vectorstore = Chroma(
            persist_directory=persist_dir,
            embedding_function=embeddings,
            collection_name=collection_name,
        )
        return vectorstore

    docs = make_documents(train_df)

    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_dir,
        collection_name=collection_name,
    )
    return vectorstore


# ============================================================
# RERANK DETERMINÍSTICO DOS CANDIDATOS RECUPERADOS
# ============================================================

def score_candidate(
    diagnosis: str,
    topography: str,
    snomed: str,
    candidate_meta: Dict[str, str],
    vector_distance: Optional[float] = None
) -> float:
    diag_norm = normalize_text(diagnosis)
    topo_norm = normalize_text(topography)
    snomed_norm = normalize_snomed(snomed)

    cand_diag = candidate_meta.get("diagnostico_norm", "")
    cand_topo = candidate_meta.get("topografia_norm", "")
    cand_snomed = candidate_meta.get("snomed", "")
    cand_morph = candidate_meta.get("morph_code", "")

    query_morph, query_behavior = parse_snomed_morphology(snomed_norm)
    cand_behavior = candidate_meta.get("behavior_code", "")

    score = 0.0

    # 1) SNOMED exato pesa bastante
    if snomed_norm and cand_snomed == snomed_norm:
        score += 50

    # 2) Morfologia exata ajuda quando o SNOMED completo não bate
    if query_morph and cand_morph == query_morph:
        score += 18

    # 3) Comportamento do SNOMED
    if query_behavior and cand_behavior == query_behavior:
        score += 8

    # 4) Topografia
    if topo_norm and cand_topo == topo_norm:
        score += 22
    else:
        q_topo = text_tokens(topo_norm)
        c_topo = text_tokens(cand_topo)
        if q_topo and c_topo:
            overlap = len(q_topo & c_topo) / max(len(q_topo), 1)
            score += overlap * 16

    # 5) Diagnóstico
    if diag_norm and cand_diag == diag_norm:
        score += 20
    else:
        q_diag = text_tokens(diag_norm)
        c_diag = text_tokens(cand_diag)
        if q_diag and c_diag:
            overlap = len(q_diag & c_diag) / max(len(q_diag), 1)
            score += overlap * 20

    # 6) CID explícito no texto
    cids_in_text = extract_cids(diagnosis)
    if candidate_meta.get("cid", "") in cids_in_text:
        score += 80

    # 7) Similaridade vetorial (distância menor = melhor)
    if vector_distance is not None:
        score += max(0, 10 - float(vector_distance))

    return round(score, 4)


def retrieve_and_rerank(
    diagnosis: str,
    topography: str,
    snomed: str,
    vectorstore: Chroma,
    k: int = 15
) -> List[Dict[str, object]]:

    query = f"""
Classificar CID oncológico.
Diagnóstico: {diagnosis}
Topografia: {topography}
SNOMED: {snomed}
Priorizar compatibilidade entre morfologia, comportamento e topografia.
""".strip()

    hits = vectorstore.similarity_search_with_score(query, k=k)

    candidates = []
    seen = set()

    for doc, distance in hits:
        meta = dict(doc.metadata)
        cid = meta.get("cid", "")

        if not cid:
            continue

        score = score_candidate(
            diagnosis=diagnosis,
            topography=topography,
            snomed=snomed,
            candidate_meta=meta,
            vector_distance=distance
        )

        # Mantém o melhor score por CID
        if cid in seen:
            continue

        candidates.append({
            "cid": cid,
            "score": score,
            "distance": float(distance),
            "snomed": meta.get("snomed", ""),
            "diagnostico": meta.get("diagnostico", ""),
            "topografia": meta.get("topografia", ""),
            "descricao_cid": meta.get("descricao_cid", ""),
            "classificacao_final": meta.get("classificacao_final", ""),
            "behavior_label": meta.get("behavior_label", ""),
        })
        seen.add(cid)

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    return candidates


# ============================================================
# LLM OPCIONAL PARA ESCOLHER ENTRE OS CANDIDATOS RECUPERADOS
# ============================================================

def safe_json_loads(text: str) -> Optional[dict]:
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None


def build_llm(model_name: str = LLM_MODEL) -> ChatOllama:
    return ChatOllama(model=model_name, temperature=0)


def llm_choose_candidate(
    diagnosis: str,
    topography: str,
    snomed: str,
    candidates: List[Dict[str, object]],
    llm: ChatOllama
) -> Optional[dict]:

    if not candidates:
        return None

    prompt = ChatPromptTemplate.from_template("""
Você é um classificador de CID oncológico.

Tarefa:
Escolha o CID mais compatível para o caso abaixo.

CASO
Diagnóstico: {diagnosis}
Topografia: {topography}
SNOMED: {snomed}

CANDIDATOS RECUPERADOS
{candidates_json}

REGRAS
1. Priorize SNOMED exato.
2. Depois priorize morfologia SNOMED + topografia compatível.
3. Use o diagnóstico apenas como apoio sem inventar CID.
4. Escolha SOMENTE entre os candidatos fornecidos.
5. Se todos os candidatos estiverem ruins, retorne "INDET".

Responda SOMENTE em JSON válido:
{{
  "cid": "C50",
  "confidence": 0.0,
  "reason": "motivo curto"
}}
""")

    chain = prompt | llm | StrOutputParser()

    raw = chain.invoke({
        "diagnosis": diagnosis,
        "topography": topography,
        "snomed": snomed,
        "candidates_json": json.dumps(candidates[:8], ensure_ascii=False, indent=2)
    })

    parsed = safe_json_loads(raw)
    return parsed


# ============================================================
# CLASSIFICAÇÃO FINAL
# ============================================================

def classify_case(
    diagnosis: str,
    topography: str,
    snomed: str,
    kb: Dict[str, object],
    vectorstore: Chroma,
    llm: Optional[ChatOllama] = None,
    use_llm: bool = True
) -> Dict[str, object]:

    diagnosis = "" if diagnosis is None else str(diagnosis)
    topography = "" if topography is None else str(topography)
    snomed = "" if snomed is None else str(snomed)

    diag_norm = normalize_text(diagnosis)
    topo_norm = normalize_text(topography)
    snomed_norm = normalize_snomed(snomed)

    # --------------------------------------------------------
    # 1) Se o CID já vier explícito no texto, usa primeiro
    # --------------------------------------------------------
    explicit_cids = extract_cids(diagnosis) | extract_cids(topography)
    for cid in explicit_cids:
        if cid in kb["cid_lookup"]:
            info = kb["cid_lookup"][cid]
            return {
                "cid_predito": cid,
                "descricao_cid": info.get("descricao_cid", ""),
                "classificacao_final": info.get("classificacao_final", ""),
                "metodo": "cid_explicito_no_texto",
                "confidence": 1.0,
                "motivo": "CID já presente no texto de entrada."
            }

    # --------------------------------------------------------
    # 2) Regra exata: SNOMED + topografia
    # --------------------------------------------------------
    cid_exact = kb["exact_snomed_topo"].get((snomed_norm, topo_norm))
    if cid_exact:
        info = kb["cid_lookup"].get(cid_exact, {})
        return {
            "cid_predito": cid_exact,
            "descricao_cid": info.get("descricao_cid", ""),
            "classificacao_final": info.get("classificacao_final", ""),
            "metodo": "regra_exata_snomed_topografia",
            "confidence": 0.99,
            "motivo": "Casamento exato entre SNOMED e topografia."
        }

    # --------------------------------------------------------
    # 3) Regra exata: diagnóstico + topografia
    # --------------------------------------------------------
    cid_exact_diag = kb["exact_diag_topo"].get((diag_norm, topo_norm))
    if cid_exact_diag:
        info = kb["cid_lookup"].get(cid_exact_diag, {})
        return {
            "cid_predito": cid_exact_diag,
            "descricao_cid": info.get("descricao_cid", ""),
            "classificacao_final": info.get("classificacao_final", ""),
            "metodo": "regra_exata_diagnostico_topografia",
            "confidence": 0.97,
            "motivo": "Casamento exato entre diagnóstico e topografia."
        }

    # --------------------------------------------------------
    # 4) Regra SNOMED sozinho, se houver maioria muito clara
    # --------------------------------------------------------
    if snomed_norm in kb["snomed_majority"]:
        counts = kb["snomed_majority"][snomed_norm]
        total = sum(v for _, v in counts)
        best_cid, best_n = counts[0]
        dominance = best_n / total if total else 0

        if dominance >= 0.85:
            info = kb["cid_lookup"].get(best_cid, {})
            return {
                "cid_predito": best_cid,
                "descricao_cid": info.get("descricao_cid", ""),
                "classificacao_final": info.get("classificacao_final", ""),
                "metodo": "regra_snomed_majoritario",
                "confidence": round(dominance, 4),
                "motivo": f"SNOMED com CID majoritário claro na base ({best_n}/{total})."
            }

    # --------------------------------------------------------
    # 5) Recuperação vetorial + rerank
    # --------------------------------------------------------
    candidates = retrieve_and_rerank(
        diagnosis=diagnosis,
        topography=topography,
        snomed=snomed,
        vectorstore=vectorstore,
        k=15
    )

    if not candidates:
        return {
            "cid_predito": "INDET",
            "descricao_cid": "",
            "classificacao_final": "",
            "metodo": "sem_candidatos",
            "confidence": 0.0,
            "motivo": "Nenhum candidato foi recuperado pela base vetorial."
        }

    # --------------------------------------------------------
    # 6) Escolha via LLM (opcional)
    # --------------------------------------------------------
    chosen_cid = None
    llm_reason = ""

    if use_llm and llm is not None:
        llm_answer = llm_choose_candidate(
            diagnosis=diagnosis,
            topography=topography,
            snomed=snomed,
            candidates=candidates,
            llm=llm
        )

        if llm_answer:
            llm_cid = normalize_cid(llm_answer.get("cid", ""))
            if llm_cid == "INDET":
                chosen_cid = "INDET"
                llm_reason = llm_answer.get("reason", "")
            elif any(c["cid"] == llm_cid for c in candidates):
                chosen_cid = llm_cid
                llm_reason = llm_answer.get("reason", "")

    # --------------------------------------------------------
    # 7) Fallback: melhor candidato pelo rerank
    # --------------------------------------------------------
    if not chosen_cid:
        chosen_cid = candidates[0]["cid"]
        llm_reason = "Melhor candidato pelo reranqueamento determinístico."

    if chosen_cid == "INDET":
        return {
            "cid_predito": "INDET",
            "descricao_cid": "",
            "classificacao_final": "",
            "metodo": "rag_llm_indeterminado",
            "confidence": 0.0,
            "motivo": llm_reason or "LLM considerou os candidatos insuficientes."
        }

    info = kb["cid_lookup"].get(chosen_cid, {})
    top_score = candidates[0]["score"] if candidates else 0.0
    conf = min(0.95, max(0.20, top_score / 100))

    return {
        "cid_predito": chosen_cid,
        "descricao_cid": info.get("descricao_cid", candidates[0].get("descricao_cid", "")),
        "classificacao_final": info.get("classificacao_final", candidates[0].get("classificacao_final", "")),
        "metodo": "rag_llm" if use_llm and llm is not None else "rag_rerank",
        "confidence": round(conf, 4),
        "motivo": llm_reason,
        "top_5_candidatos": candidates[:5]
    }


# ============================================================
# CLASSIFICAÇÃO EM LOTE
# ============================================================

def classify_dataframe(
    df_cases: pd.DataFrame,
    kb: Dict[str, object],
    vectorstore: Chroma,
    llm: Optional[ChatOllama] = None,
    diagnosis_col: str = "Diagnóstico",
    topography_col: str = "Topografia",
    snomed_col: str = "SNOMED",
    use_llm: bool = True,
    checkpoint_path: Optional[str] = "checkpoint_resultado_snomed_cid.csv",
    checkpoint_every: int = 20
) -> pd.DataFrame:

    outputs = []

    rows = df_cases.to_dict("records")

    for i, row in enumerate(tqdm(rows, total=len(rows), desc="Classificando casos"), start=1):
        result = classify_case(
            diagnosis=row.get(diagnosis_col, ""),
            topography=row.get(topography_col, ""),
            snomed=row.get(snomed_col, ""),
            kb=kb,
            vectorstore=vectorstore,
            llm=llm,
            use_llm=use_llm
        )

        outputs.append(result)

        if checkpoint_path and (i % checkpoint_every == 0):
            parcial = pd.concat(
                [df_cases.iloc[:i].reset_index(drop=True), pd.DataFrame(outputs)],
                axis=1
            )
            parcial.to_csv(checkpoint_path, index=False, sep=";", encoding="utf-8-sig")
            print(f"\nCheckpoint salvo com {i} casos.")
            print(parcial[[
                "Diagnóstico", "Topografia", "SNOMED",
                "cid_predito", "descricao_cid",
                "metodo", "confidence"
            ]].tail(5).to_string(index=False))

    df_final = pd.concat(
        [df_cases.reset_index(drop=True), pd.DataFrame(outputs)],
        axis=1
    )
    df_final.to_csv("Resultado_SNOMED_CID_Parcial_predito.csv", sep=";", index=False, encoding="utf-8-sig")

    if checkpoint_path:
        df_final.to_csv(checkpoint_path, index=False, sep=";", encoding="utf-8-sig")

    return df_final


# ============================================================
# EXEMPLO DE USO
# ============================================================

if __name__ == "__main__":
    kb = load_knowledge_bases(
        base_cid_file=BASE_CID_FILE,
        snomed_cid_file=SNOMED_CID_FILE
    )

    vectorstore = build_vectorstore(
        train_df=kb["train_df"],
        persist_dir=PERSIST_DIR,
        collection_name=COLLECTION_NAME,
        embed_model=EMBED_MODEL
    )

    # Opcional: LLM para decidir entre os candidatos recuperados
    llm = build_llm(LLM_MODEL)

    exemplo = classify_case(
        diagnosis="adenocarcinoma ductal invasivo",
        topography="mama",
        snomed="M-85003",
        kb=kb,
        vectorstore=vectorstore,
        llm=llm,
        use_llm=True
    )

    print(json.dumps(exemplo, ensure_ascii=False, indent=2))

    # Exemplo em lote
    casos = read_csv_flex("Resultado_SNOMED_CID_Parcial - Copia.csv")
    
    # pd.DataFrame([
    #     {
    #         "Diagnóstico": "adenocarcinoma ductal invasivo",
    #         "Topografia": "mama",
    #         "SNOMED": "M-85003"
    #     },
    #     {
    #         "Diagnóstico": "adenocarcinoma acinar usual",
    #         "Topografia": "próstata",
    #         "SNOMED": "M-81403"
    #     },
    #     {
    #         "Diagnóstico": "carcinoma espinocelular invasivo",
    #         "Topografia": "colo uterino",
    #         "SNOMED": "M-80703"
    #     }
    # ])

    df_out = classify_dataframe(
        df_cases=casos,
        kb=kb,
        vectorstore=vectorstore,
        llm=llm,
        diagnosis_col="Diagnóstico",
        topography_col="Topografia",
        snomed_col="SNOMED",
        use_llm=True
    )

    print(df_out[[
        "Diagnóstico", "Topografia", "SNOMED",
        "cid_predito", "descricao_cid",
        "classificacao_final", "metodo", "confidence"
    ]])


    print("4. Iniciando processamento híbrido...")



{
  "cid_predito": "C50",
  "descricao_cid": "Neoplasia maligna da mama",
  "classificacao_final": "CR Tumores da Mama",
  "metodo": "regra_exata_snomed_topografia",
  "confidence": 0.99,
  "motivo": "Casamento exato entre SNOMED e topografia."
}


Classificando casos:   0%|          | 0/119 [00:00<?, ?it/s]


Checkpoint salvo com 20 casos.
              Diagnóstico               Topografia  SNOMED cid_predito             descricao_cid                   metodo  confidence
Adenocarcinoma Em Adenoma         Cólon ascendente M-82103         C50 Neoplasia maligna da mama regra_snomed_majoritario         1.0
Adenocarcinoma Em Adenoma         Cólon ascendente M-82103         C50 Neoplasia maligna da mama regra_snomed_majoritario         1.0
Adenocarcinoma Em Adenoma         Cólon ascendente M-82103         C50 Neoplasia maligna da mama regra_snomed_majoritario         1.0
Adenocarcinoma Em Adenoma Pólipo de cólon sigmoide M-82103         C50 Neoplasia maligna da mama regra_snomed_majoritario         1.0
Adenocarcinoma Em Adenoma Pólipo de cólon sigmoide M-82103         C50 Neoplasia maligna da mama regra_snomed_majoritario         1.0

Checkpoint salvo com 40 casos.
        Diagnóstico                 Topografia  SNOMED cid_predito                                          descricao_cid  metodo  c